# ⚽ FutStore Brasil — Dashboard Executivo de Vendas
## ETL · Análise Exploratória · Visualização de KPIs

**Fonte:** Dataset proprietário gerado para simulação de uma loja de artigos esportivos de futebol  
**Período:** Janeiro – Junho/2024  
**Ferramenta:** Python (Pandas, Matplotlib, Seaborn, Plotly)


## 0. Instalação e Importação de Bibliotecas

In [ ]:
# Instalação (caso necessário no Colab)
# !pip install plotly --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuração visual padrão
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5), 'font.size': 11})
print("✅ Bibliotecas carregadas com sucesso!")


## 1. EXTRAÇÃO — Carregamento dos Dados

In [ ]:
# ── Carregue o arquivo CSV no mesmo diretório ou ajuste o caminho ──
df_raw = pd.read_csv('futstore_vendas.csv')

print(f"Shape inicial: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
df_raw.head(8)


In [ ]:
# Informações gerais do dataset
df_raw.info()


In [ ]:
# Estatísticas descritivas
df_raw.describe()


## 2. TRANSFORMAÇÃO — Limpeza e Enriquecimento dos Dados

### 2.1 Identificação e Remoção de Duplicatas

In [ ]:
qtd_dup = df_raw.duplicated().sum()
print(f"🔍 Linhas duplicadas encontradas: {qtd_dup}")
df_raw[df_raw.duplicated(keep=False)]


In [ ]:
df = df_raw.drop_duplicates()
print(f"✅ Duplicatas removidas. Linhas restantes: {len(df)}")


### 2.2 Tratamento de Valores Ausentes (Missing Values)

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Nulos': missing, '% do Total': missing_pct})
print(missing_df[missing_df['Nulos'] > 0])

# Verifica também strings vazias que não são NaN
vazios = (df == '').sum()
print("\nCampos com string vazia:")
print(vazios[vazios > 0])


In [ ]:
# Substitui strings vazias por NaN e então preenche com 'Desconhecido'
df['nome_cliente'] = df['nome_cliente'].replace('', np.nan)
df['nome_cliente'] = df['nome_cliente'].fillna('Desconhecido')
print(f"✅ Valores ausentes em nome_cliente tratados.")
print(df[df['nome_cliente'] == 'Desconhecido'][['id_venda','nome_cliente','produto','categoria']])


### 2.3 Conversão de Tipos de Dados

In [ ]:
# Converte 'data' de string para datetime
df['data'] = pd.to_datetime(df['data'])

# Cria colunas derivadas de data
df['mes']        = df['data'].dt.month
df['mes_nome']   = df['data'].dt.strftime('%b/%Y')
df['trimestre']  = df['data'].dt.to_period('Q').astype(str)
df['dia_semana'] = df['data'].dt.day_name()

# Garante tipos numéricos corretos
df['preco_unitario'] = df['preco_unitario'].astype(float)
df['custo_unitario'] = df['custo_unitario'].astype(float)
df['quantidade']     = df['quantidade'].astype(int)

# Cria colunas calculadas de negócio
df['total_venda']  = df['preco_unitario'] * df['quantidade']
df['total_custo']  = df['custo_unitario'] * df['quantidade']
df['lucro']        = df['total_venda'] - df['total_custo']
df['margem_pct']   = (df['lucro'] / df['total_venda'] * 100).round(2)

print("✅ Tipos convertidos e colunas de negócio criadas.")
print(df.dtypes)


In [ ]:
# Visão final do dataset limpo
print(f"Dataset final: {df.shape[0]} linhas × {df.shape[1]} colunas")
df.head(5)


## 3. ANÁLISE DE GESTÃO — KPIs de Negócio

### 📌 KPI 1 — Ticket Médio por Categoria

In [ ]:
ticket_medio = (
    df.groupby('categoria')['total_venda']
      .mean()
      .sort_values(ascending=False)
      .round(2)
      .reset_index()
      .rename(columns={'total_venda': 'Ticket Médio (R$)'})
)
print(ticket_medio.to_string(index=False))


### 📌 KPI 2 — Margem de Lucro por Categoria

In [ ]:
margem_cat = (
    df.groupby('categoria')
      .apply(lambda x: (x['lucro'].sum() / x['total_venda'].sum() * 100).round(2))
      .reset_index(name='Margem de Lucro (%)')
      .sort_values('Margem de Lucro (%)', ascending=False)
)
print(margem_cat.to_string(index=False))


### 📌 KPI 3 — Sazonalidade: Faturamento Mensal

In [ ]:
ordem_meses = ['Jan/2024','Feb/2024','Mar/2024','Apr/2024','May/2024','Jun/2024']

sazonalidade = (
    df.groupby('mes_nome')[['total_venda','lucro']]
      .sum()
      .round(2)
      .reindex(ordem_meses)
      .reset_index()
      .rename(columns={'mes_nome':'Mês', 'total_venda':'Faturamento (R$)', 'lucro':'Lucro (R$)'})
)
sazonalidade['Margem (%)'] = (sazonalidade['Lucro (R$)'] / sazonalidade['Faturamento (R$)'] * 100).round(2)
print(sazonalidade.to_string(index=False))


### 📌 KPI 4 — Top 10 Clientes por Faturamento

In [ ]:
top10 = (
    df[df['nome_cliente'] != 'Desconhecido']
      .groupby('nome_cliente')['total_venda']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .round(2)
      .reset_index()
      .rename(columns={'nome_cliente':'Cliente', 'total_venda':'Total Gasto (R$)'})
)
top10.index = top10.index + 1
print(top10.to_string())


### 📌 KPI 5 — Performance por Vendedor

In [ ]:
vendedor_kpi = (
    df.groupby('vendedor')
      .agg(
          Faturamento   = ('total_venda', 'sum'),
          Lucro         = ('lucro', 'sum'),
          Num_Vendas    = ('id_venda', 'count'),
          Ticket_Medio  = ('total_venda', 'mean')
      )
      .round(2)
      .sort_values('Faturamento', ascending=False)
      .reset_index()
)
print(vendedor_kpi.to_string(index=False))


## 4. VISUALIZAÇÃO DE DADOS — Dashboard Executivo

### 📊 Gráfico 1 — Faturamento Mensal (Sazonalidade)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

cores = ['#e63946' if v == sazonalidade['Faturamento (R$)'].max() else '#1d3557'
         for v in sazonalidade['Faturamento (R$)']]

bars = ax.bar(sazonalidade['Mês'], sazonalidade['Faturamento (R$)'],
              color=cores, edgecolor='white', width=0.6)

# Rótulos nas barras
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 200,
            f"R$ {bar.get_height():,.0f}",
            ha='center', va='bottom', fontsize=9.5, fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.set_title('⚽ FutStore Brasil — Faturamento Mensal (Jan–Jun/2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mês', fontsize=11)
ax.set_ylabel('Faturamento (R$)', fontsize=11)
ax.set_ylim(0, sazonalidade['Faturamento (R$)'].max() * 1.18)
ax.legend(handles=[plt.Rectangle((0,0),1,1, color='#e63946'), plt.Rectangle((0,0),1,1, color='#1d3557')],
          labels=['Maior mês','Demais meses'], loc='upper right')

plt.tight_layout()
plt.savefig('grafico1_faturamento_mensal.png', bbox_inches='tight')
plt.show()


### 📊 Gráfico 2 — Ticket Médio e Margem por Categoria

In [ ]:
# Combinar dados de ticket e margem por categoria
cat_resumo = df.groupby('categoria').agg(
    ticket_medio  = ('total_venda', 'mean'),
    margem        = ('margem_pct', 'mean')
).round(2).sort_values('ticket_medio', ascending=True).reset_index()

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

bars = ax1.barh(cat_resumo['categoria'], cat_resumo['ticket_medio'],
                color='#457b9d', alpha=0.85, label='Ticket Médio (R$)')
line = ax2.plot(cat_resumo['margem'], cat_resumo['categoria'],
                'o-', color='#e63946', linewidth=2.5, markersize=8, label='Margem (%)')

for bar, val in zip(bars, cat_resumo['ticket_medio']):
    ax1.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
             f'R$ {val:.0f}', va='center', fontsize=9)

ax1.set_xlabel('Ticket Médio (R$)', fontsize=11)
ax2.set_xlabel('Margem (%)', fontsize=11)
ax2.set_ylabel('Margem de Lucro (%)', fontsize=11, color='#e63946')
ax2.tick_params(axis='y', labelcolor='#e63946')
ax1.set_title('Ticket Médio vs. Margem de Lucro por Categoria', fontsize=13, fontweight='bold', pad=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

plt.tight_layout()
plt.savefig('grafico2_ticket_margem_categoria.png', bbox_inches='tight')
plt.show()


### 📊 Gráfico 3 — Distribuição de Vendas por Time

In [ ]:
vendas_time = (df.groupby('time')['total_venda']
                 .sum()
                 .sort_values(ascending=False)
                 .reset_index())

fig, ax = plt.subplots(figsize=(11, 5))
palette = sns.color_palette('tab10', n_colors=len(vendas_time))
sns.barplot(data=vendas_time, x='time', y='total_venda',
            palette=palette, ax=ax, edgecolor='white')

for i, row in vendas_time.iterrows():
    ax.text(i, row['total_venda'] + 100, f"R$ {row['total_venda']:,.0f}",
            ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.set_title('Faturamento Total por Time (Jan–Jun/2024)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Faturamento Total (R$)', fontsize=11)
ax.tick_params(axis='x', rotation=20)
ax.set_ylim(0, vendas_time['total_venda'].max() * 1.18)

plt.tight_layout()
plt.savefig('grafico3_vendas_por_time.png', bbox_inches='tight')
plt.show()


### 📊 Gráfico 4 — Top 10 Clientes (Faturamento)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

cores_top10 = ['#e63946' if i == 0 else '#1d3557' for i in range(len(top10))]
bars = ax.barh(top10['Cliente'][::-1], top10['Total Gasto (R$)'][::-1],
               color=cores_top10[::-1], edgecolor='white')

for bar in bars:
    ax.text(bar.get_width() + 10,
            bar.get_y() + bar.get_height()/2,
            f"R$ {bar.get_width():,.0f}",
            va='center', fontsize=9.5, fontweight='bold')

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.set_title('🏆 Top 10 Clientes por Faturamento', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Total Gasto (R$)', fontsize=11)
ax.set_ylabel('Cliente', fontsize=11)
ax.set_xlim(0, top10['Total Gasto (R$)'].max() * 1.22)
ax.legend(handles=[plt.Rectangle((0,0),1,1, color='#e63946')], labels=['#1 Cliente'], loc='lower right')

plt.tight_layout()
plt.savefig('grafico4_top10_clientes.png', bbox_inches='tight')
plt.show()


### 📊 Gráfico 5 — Forma de Pagamento (Participação %)

In [ ]:
pag_share = df.groupby('forma_pagamento')['total_venda'].sum().sort_values(ascending=False)

cores_pizza = ['#1d3557','#457b9d','#a8dadc','#e63946','#f4a261']
explode    = [0.05 if i == 0 else 0 for i in range(len(pag_share))]

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    pag_share.values,
    labels=pag_share.index,
    autopct='%1.1f%%',
    colors=cores_pizza,
    explode=explode,
    startangle=140,
    pctdistance=0.82,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for text in autotexts:
    text.set_fontsize(11)
    text.set_fontweight('bold')
    text.set_color('white')

ax.set_title('Participação por Forma de Pagamento\n(% do Faturamento Total)',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(wedges, [f"{k}: R$ {v:,.0f}" for k, v in zip(pag_share.index, pag_share.values)],
          title="Faturamento", loc="lower left", bbox_to_anchor=(0.0, -0.05), fontsize=9)

plt.tight_layout()
plt.savefig('grafico5_formas_pagamento.png', bbox_inches='tight')
plt.show()


### 📊 Gráfico 6 (Bônus) — Heatmap de Vendas: Categoria × Região

In [ ]:
heatmap_data = df.pivot_table(
    values='total_venda', index='categoria', columns='regiao', aggfunc='sum', fill_value=0
).round(0)

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=',.0f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Faturamento (R$)'}
)
ax.set_title('Heatmap — Faturamento por Categoria × Região', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Região', fontsize=11)
ax.set_ylabel('Categoria', fontsize=11)
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('grafico6_heatmap_categoria_regiao.png', bbox_inches='tight')
plt.show()


## 5. RESUMO EXECUTIVO — Principais Insights

In [ ]:
fat_total   = df['total_venda'].sum()
lucro_total = df['lucro'].sum()
margem_geral= lucro_total / fat_total * 100
ticket_geral= df.groupby('id_venda')['total_venda'].sum().mean()
melhor_cat  = df.groupby('categoria')['total_venda'].sum().idxmax()
melhor_time = df.groupby('time')['total_venda'].sum().idxmax()
melhor_mes  = sazonalidade.loc[sazonalidade['Faturamento (R$)'].idxmax(), 'Mês']
melhor_vend = vendedor_kpi.iloc[0]['vendedor']
top_cliente = top10.iloc[0]['Cliente']

print("=" * 52)
print("     ⚽  FUTSTORE BRASIL — RESUMO EXECUTIVO  ⚽    ")
print("=" * 52)
print(f"  Período Analisado : Jan – Jun/2024")
print(f"  Total de Vendas   : {len(df)} transações")
print(f"  Faturamento Total : R$ {fat_total:>10,.2f}")
print(f"  Lucro Bruto Total : R$ {lucro_total:>10,.2f}")
print(f"  Margem Geral      : {margem_geral:>9.1f}%")
print(f"  Ticket Médio      : R$ {ticket_geral:>10,.2f}")
print("-" * 52)
print(f"  Categoria Top     : {melhor_cat}")
print(f"  Time Top          : {melhor_time}")
print(f"  Mês de Pico       : {melhor_mes}")
print(f"  Melhor Vendedor   : {melhor_vend}")
print(f"  Cliente VIP #1    : {top_cliente}")
print("=" * 52)
